In [1]:
import os
print("Kaggle working directory:", os.listdir("/kaggle/working"))
print("\nKaggle input datasets:", os.listdir("/kaggle/input"))
for dataset in os.listdir("/kaggle/input"):
    dataset_path = os.path.join("/kaggle/input", dataset)
    print(f"\nDataset: {dataset}")
    print("Contents:", os.listdir(dataset_path))

Kaggle working directory: []

Kaggle input datasets: ['datasets']

Dataset: datasets
Contents: ['bhoomi0805']


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root).lower() == "color":
        print("COLOR DATASET FOUND:")
        print(root)
        print("\nFirst 10 class folders:")
        print(os.listdir(root)[:10])

COLOR DATASET FOUND:
/kaggle/input/datasets/bhoomi0805/plant-village/color

First 10 class folders:
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight']


In [3]:
import os

COLOR_PATH = "/kaggle/input/datasets/bhoomi0805/plant-village/color"

tomato_classes = sorted([
    folder for folder in os.listdir(COLOR_PATH)
    if folder.startswith("Tomato___")
    and os.path.isdir(os.path.join(COLOR_PATH, folder))
])

print("Number of tomato classes:", len(tomato_classes))
print("\nTomato classes and image counts:\n")

total_images = 0

for class_name in tomato_classes:
    class_path = os.path.join(COLOR_PATH, class_name)

    count = len([
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    total_images += count
    print(f"{class_name}: {count}")

print("\nTotal tomato images:", total_images)

Number of tomato classes: 10

Tomato classes and image counts:

Tomato___Bacterial_spot: 2127
Tomato___Early_blight: 1000
Tomato___Late_blight: 1405
Tomato___Leaf_Mold: 456
Tomato___Septoria_leaf_spot: 970
Tomato___Spider_mites Two-spotted_spider_mite: 1126
Tomato___Target_Spot: 1012
Tomato___Tomato_Yellow_Leaf_Curl_Virus: 4154
Tomato___Tomato_mosaic_virus: 165
Tomato___healthy: 1591

Total tomato images: 14006


In [4]:
from PIL import Image
from tqdm import tqdm

corrupted_images = []
verified_count = 0

for class_name in tqdm(tomato_classes, desc="Checking tomato images"):
    class_path = os.path.join(COLOR_PATH, class_name)

    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(class_path, filename)

            try:
                with Image.open(image_path) as img:
                    img.verify()
                verified_count += 1

            except Exception:
                corrupted_images.append(image_path)

print("\nImage integrity report")
print("Verified images:", verified_count)
print("Corrupted images:", len(corrupted_images))

Checking tomato images: 100%|██████████| 10/10 [12:56<00:00, 77.62s/it]


Image integrity report
Verified images: 14006
Corrupted images: 0


In [5]:
import hashlib
from collections import defaultdict
from tqdm import tqdm

hash_to_images = defaultdict(list)

for class_name in tqdm(tomato_classes, desc="Hashing images"):
    class_path = os.path.join(COLOR_PATH, class_name)

    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(class_path, filename)

            with open(image_path, "rb") as f:
                image_hash = hashlib.md5(f.read()).hexdigest()

            hash_to_images[image_hash].append({
                "path": image_path,
                "class": class_name
            })

duplicate_groups = {
    h: images
    for h, images in hash_to_images.items()
    if len(images) > 1
}

duplicate_copies = sum(
    len(images) - 1
    for images in duplicate_groups.values()
)

cross_class_groups = sum(
    len(set(item["class"] for item in images)) > 1
    for images in duplicate_groups.values()
)

print("Total image files:", sum(len(v) for v in hash_to_images.values()))
print("Unique image hashes:", len(hash_to_images))
print("Duplicate groups:", len(duplicate_groups))
print("Extra duplicate copies:", duplicate_copies)
print("Cross-class duplicate groups:", cross_class_groups)

Hashing images: 100%|██████████| 10/10 [02:59<00:00, 17.96s/it]

Total image files: 14006
Unique image hashes: 13995
Duplicate groups: 11
Extra duplicate copies: 11
Cross-class duplicate groups: 0


In [6]:
import pandas as pd

manifest_records = []

for image_hash, images in tqdm(hash_to_images.items(), desc="Creating manifest"):

    # Keep one copy of each exact image
    selected_image = images[0]

    manifest_records.append({
        "image_path": selected_image["path"],
        "class_name": selected_image["class"],
        "image_hash": image_hash
    })

df = pd.DataFrame(manifest_records)

print("Clean manifest created!")
print("Total unique images:", len(df))
print("\nImages per class:")
print(df["class_name"].value_counts().sort_index())

print("\nDuplicate copies excluded:", 14006 - len(df))

Creating manifest: 100%|██████████| 13995/13995 [00:00<00:00, 1250623.92it/s]

Clean manifest created!
Total unique images: 13995

Images per class:
class_name
Tomato___Bacterial_spot                          2127
Tomato___Early_blight                            1000
Tomato___Late_blight                             1400
Tomato___Leaf_Mold                                456
Tomato___Septoria_leaf_spot                       970
Tomato___Spider_mites Two-spotted_spider_mite    1126
Tomato___Target_Spot                             1012
Tomato___Tomato_Yellow_Leaf_Curl_Virus           4154
Tomato___Tomato_mosaic_virus                      165
Tomato___healthy                                 1585
Name: count, dtype: int64

Duplicate copies excluded: 11


In [7]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["class_name"]
)

# Second split: divide temporary set equally into validation and test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["class_name"]
)

print("Dataset split completed!\n")

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nTotal:", len(train_df) + len(val_df) + len(test_df))

Dataset split completed!

Training images: 9796
Validation images: 2099
Testing images: 2100

Total: 13995


In [8]:
train_hashes = set(train_df["image_hash"])
val_hashes = set(val_df["image_hash"])
test_hashes = set(test_df["image_hash"])

print("Train-Val overlap:", len(train_hashes & val_hashes))
print("Train-Test overlap:", len(train_hashes & test_hashes))
print("Val-Test overlap:", len(val_hashes & test_hashes))

all_hashes = train_hashes | val_hashes | test_hashes

print("\nTotal unique hashes across splits:", len(all_hashes))
print("Original clean dataset size:", len(df))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

Total unique hashes across splits: 13995
Original clean dataset size: 13995


In [10]:
import os

PROJECT_DIR = "/kaggle/working/PlantDiseaseProject"
os.makedirs(PROJECT_DIR, exist_ok=True)

train_df.to_csv(
    os.path.join(PROJECT_DIR, "train_manifest.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(PROJECT_DIR, "validation_manifest.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(PROJECT_DIR, "test_manifest.csv"),
    index=False
)

print("All manifests saved successfully!")

print("\nSaved files:")
print(os.listdir(PROJECT_DIR))

All manifests saved successfully!

Saved files:
['validation_manifest.csv', 'train_manifest.csv', 'test_manifest.csv']


In [11]:
import numpy as np

class_counts = (
    train_df["class_name"]
    .value_counts()
    .sort_index()
)

num_classes = len(class_counts)
total_train = len(train_df)

class_weights = total_train / (num_classes * class_counts)

class_distribution = pd.DataFrame({
    "Class": class_counts.index,
    "Training Images": class_counts.values,
    "Class Weight": class_weights.values
})

print("Training class distribution and weights:\n")
print(class_distribution.to_string(index=False))

print("\nLargest class:", class_counts.max())
print("Smallest class:", class_counts.min())
print("Imbalance ratio:", round(class_counts.max() / class_counts.min(), 2))

Training class distribution and weights:

                                        Class  Training Images  Class Weight
                      Tomato___Bacterial_spot             1489      0.657891
                        Tomato___Early_blight              700      1.399429
                         Tomato___Late_blight              980      0.999592
                           Tomato___Leaf_Mold              319      3.070846
                  Tomato___Septoria_leaf_spot              679      1.442710
Tomato___Spider_mites Two-spotted_spider_mite              788      1.243147
                         Tomato___Target_Spot              708      1.383616
       Tomato___Tomato_Yellow_Leaf_Curl_Virus             2908      0.336864
                 Tomato___Tomato_mosaic_virus              116      8.444828
                             Tomato___healthy             1109      0.883318

Largest class: 2908
Smallest class: 116
Imbalance ratio: 25.07


In [12]:
import json
import torch

# Sorted class mapping
class_names = sorted(train_df["class_name"].unique())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

# Convert class weights to PyTorch tensor
weights_tensor = torch.tensor(
    [class_weights[class_name] for class_name in class_names],
    dtype=torch.float32
)

# Save class mapping
with open(os.path.join(PROJECT_DIR, "class_mapping.json"), "w") as f:
    json.dump(class_to_idx, f, indent=4)

# Save class weights
torch.save(
    weights_tensor,
    os.path.join(PROJECT_DIR, "class_weights.pt")
)

print("Class mapping:")
for name, idx in class_to_idx.items():
    print(f"{idx}: {name}")

print("\nClass weights tensor:")
print(weights_tensor)

print("\nNumber of classes:", len(class_names))
print("Saved mapping and weights successfully!")

Class mapping:
0: Tomato___Bacterial_spot
1: Tomato___Early_blight
2: Tomato___Late_blight
3: Tomato___Leaf_Mold
4: Tomato___Septoria_leaf_spot
5: Tomato___Spider_mites Two-spotted_spider_mite
6: Tomato___Target_Spot
7: Tomato___Tomato_Yellow_Leaf_Curl_Virus
8: Tomato___Tomato_mosaic_virus
9: Tomato___healthy

Class weights tensor:
tensor([0.6579, 1.3994, 0.9996, 3.0708, 1.4427, 1.2431, 1.3836, 0.3369, 8.4448,
        0.8833])

Number of classes: 10
Saved mapping and weights successfully!


In [13]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("Training transformations:")
print(train_transform)

print("\nEvaluation transformations:")
print(eval_transform)

Training transformations:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.85, 1.15), contrast=(0.85, 1.15), saturation=(0.85, 1.15), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Evaluation transformations:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [14]:
from torch.utils.data import Dataset
from PIL import Image

class TomatoDataset(Dataset):

    def __init__(self, dataframe, class_to_idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.iloc[index]["image_path"]
        class_name = self.dataframe.iloc[index]["class_name"]

        image = Image.open(image_path).convert("RGB")
        label = self.class_to_idx[class_name]

        if self.transform:
            image = self.transform(image)

        return image, label


print("TomatoDataset class created successfully!")

TomatoDataset class created successfully!


In [17]:
train_dataset = TomatoDataset(
    dataframe=train_df,
    class_to_idx=class_to_idx,
    transform=train_transform
)

val_dataset = TomatoDataset(
    dataframe=val_df,
    class_to_idx=class_to_idx,
    transform=eval_transform
)

test_dataset = TomatoDataset(
    dataframe=test_df,
    class_to_idx=class_to_idx,
    transform=eval_transform
)

print("Datasets created successfully!")
print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

Datasets created successfully!
Training dataset: 9796
Validation dataset: 2099
Testing dataset: 2100


In [18]:
sample_image, sample_label = train_dataset[0]

print("Image shape:", sample_image.shape)
print("Label index:", sample_label)
print("Class name:", idx_to_class[sample_label])
print("Image data type:", sample_image.dtype)
print("Minimum pixel value:", sample_image.min().item())
print("Maximum pixel value:", sample_image.max().item())

Image shape: torch.Size([3, 224, 224])
Label index: 0
Class name: Tomato___Bacterial_spot
Image data type: torch.float32
Minimum pixel value: -2.1179039478302
Maximum pixel value: 0.9988012909889221


In [19]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("DataLoaders created successfully!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

DataLoaders created successfully!
Training batches: 307
Validation batches: 66
Testing batches: 66


In [20]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)

print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

print("Unique labels in batch:", torch.unique(labels).tolist())
print("Batch size:", images.size(0))

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
Unique labels in batch: [0, 1, 2, 3, 4, 5, 6, 7]
Batch size: 32


In [22]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("Training will run on CPU.")

PyTorch version: 2.11.0+cu130
CUDA available: True
Selected device: cuda
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [24]:
import torch

for i in range(torch.cuda.device_count()):
    torch.cuda.set_device(i)

    total_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    allocated_memory = torch.cuda.memory_allocated(i) / (1024**3)
    reserved_memory = torch.cuda.memory_reserved(i) / (1024**3)

    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Total memory: {total_memory:.2f} GB")
    print(f"Allocated memory: {allocated_memory:.2f} GB")
    print(f"Reserved memory: {reserved_memory:.2f} GB")


GPU 0: Tesla T4
Total memory: 14.56 GB
Allocated memory: 0.00 GB
Reserved memory: 0.00 GB

GPU 1: Tesla T4
Total memory: 14.56 GB
Allocated memory: 0.00 GB
Reserved memory: 0.00 GB


In [25]:
import torch
import torch.nn as nn

class CustomCNN(nn.Module):

    def __init__(self, num_classes=10):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CustomCNN(num_classes=len(class_names)).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(model)
print("\nTotal parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Model device:", next(model.parameters()).device)

CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, af

In [26]:
device = torch.device("cuda:0")

model = model.to(device)
weights_tensor = weights_tensor.to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("Training device:", device)
print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Learning rate: 0.001")
print("Weight decay: 0.0001")

Training device: cuda:0
Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)
Learning rate: 0.001
Weight decay: 0.0001


In [27]:
import os

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

NUM_EPOCHS = 10
BEST_VAL_ACCURACY = 0.0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Number of epochs:", NUM_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint system ready!")

Checkpoint directory: /kaggle/working/PlantDiseaseProject/checkpoints
Number of epochs: 10
Batch size: 32
GPU: Tesla T4
Checkpoint system ready!


In [28]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


print("Training function created successfully!")
print("Evaluation function created successfully!")

Training function created successfully!
Evaluation function created successfully!


In [29]:
from tqdm.auto import tqdm
import torch
import os

for epoch in range(NUM_EPOCHS):

    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_acc:.2f}%")

    # Save checkpoint after every epoch
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best_val_accuracy": max(BEST_VAL_ACCURACY, val_acc)
    }

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR, f"custom_cnn_epoch_{epoch + 1}.pth"
    )

    torch.save(checkpoint, checkpoint_path)

    # Save best model separately
    if val_acc > BEST_VAL_ACCURACY:

        BEST_VAL_ACCURACY = val_acc

        best_model_path = os.path.join(
            CHECKPOINT_DIR, "best_custom_cnn.pth"
        )

        torch.save(model.state_dict(), best_model_path)

        print(f"New best model saved! Validation accuracy: {BEST_VAL_ACCURACY:.2f}%")

    print(f"Checkpoint saved: {checkpoint_path}")

print("\nCustom CNN training completed!")
print(f"Best validation accuracy: {BEST_VAL_ACCURACY:.2f}%")


Epoch 1/10
Train Loss: 1.2193 | Train Accuracy: 63.40%
Val Loss:   0.9050 | Val Accuracy:   73.65%
New best model saved! Validation accuracy: 73.65%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/checkpoints/custom_cnn_epoch_1.pth

Epoch 2/10
Train Loss: 0.8338 | Train Accuracy: 74.40%
Val Loss:   0.6619 | Val Accuracy:   76.51%
New best model saved! Validation accuracy: 76.51%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/checkpoints/custom_cnn_epoch_2.pth

Epoch 3/10
Train Loss: 0.7142 | Train Accuracy: 76.81%
Val Loss:   0.5228 | Val Accuracy:   83.75%
New best model saved! Validation accuracy: 83.75%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/checkpoints/custom_cnn_epoch_3.pth

Epoch 4/10
Train Loss: 0.6338 | Train Accuracy: 80.08%
Val Loss:   0.5668 | Val Accuracy:   81.32%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/checkpoints/custom_cnn_epoch_4.pth

Epoch 5/10
Train Loss: 0.5783 | Train Accuracy: 82.03%
Val Loss:   0.4371 | Val Accuracy:   8

In [30]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import torch

# Load the best validation checkpoint
best_model_path = os.path.join(
    CHECKPOINT_DIR, "best_custom_cnn.pth"
)

model.load_state_dict(
    torch.load(best_model_path, map_location=device, weights_only=True)
)

model.to(device)
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate test metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_precision = precision_score(
    all_labels, all_predictions, average="macro", zero_division=0
)
test_recall = recall_score(
    all_labels, all_predictions, average="macro", zero_division=0
)
test_f1 = f1_score(
    all_labels, all_predictions, average="macro", zero_division=0
)

print("CUSTOM CNN — TEST RESULTS")
print("=" * 50)
print(f"Test Accuracy:  {test_accuracy * 100:.2f}%")
print(f"Macro Precision: {test_precision * 100:.2f}%")
print(f"Macro Recall:    {test_recall * 100:.2f}%")
print(f"Macro F1-Score:  {test_f1 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    all_labels,
    all_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=list(range(len(class_names)))
)

print("Confusion matrix shape:", cm.shape)
print("Test evaluation completed!")

CUSTOM CNN — TEST RESULTS
Test Accuracy:  90.48%
Macro Precision: 85.39%
Macro Recall:    90.59%
Macro F1-Score:  86.82%

CLASSIFICATION REPORT
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.96      0.92      0.94       319
                        Tomato___Early_blight       0.86      0.77      0.81       150
                         Tomato___Late_blight       0.93      0.92      0.92       210
                           Tomato___Leaf_Mold       0.87      0.96      0.91        69
                  Tomato___Septoria_leaf_spot       0.96      0.80      0.87       146
Tomato___Spider_mites Two-spotted_spider_mite       0.69      0.93      0.79       169
                         Tomato___Target_Spot       0.80      0.88      0.84       152
       Tomato___Tomato_Yellow_Leaf_Curl_Virus       0.99      0.91      0.95       623
                 Tomato___Tomato_mosaic_virus       0.50      1.00      

In [31]:
import os
import json
import pandas as pd
from sklearn.metrics import classification_report

# Save overall test metrics
metrics = {
    "model": "Custom CNN",
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(test_precision),
    "macro_recall": float(test_recall),
    "macro_f1": float(test_f1),
    "best_validation_accuracy": float(BEST_VAL_ACCURACY),
    "epochs": NUM_EPOCHS,
    "test_samples": len(all_labels)
}

with open(os.path.join(PROJECT_DIR, "custom_cnn_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

# Save per-class classification report
report = classification_report(
    all_labels,
    all_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    os.path.join(PROJECT_DIR, "custom_cnn_classification_report.csv")
)

# Save confusion matrix
pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
).to_csv(os.path.join(PROJECT_DIR, "custom_cnn_confusion_matrix.csv"))

# Save training history
pd.DataFrame(history).to_csv(
    os.path.join(PROJECT_DIR, "custom_cnn_training_history.csv"),
    index=False
)

print("Custom CNN results saved successfully!")
print("Saved files:")
print("- custom_cnn_metrics.json")
print("- custom_cnn_classification_report.csv")
print("- custom_cnn_confusion_matrix.csv")
print("- custom_cnn_training_history.csv")

Custom CNN results saved successfully!
Saved files:
- custom_cnn_metrics.json
- custom_cnn_classification_report.csv
- custom_cnn_confusion_matrix.csv
- custom_cnn_training_history.csv


In [32]:
import torchvision.models as models
from torchvision.models import ResNet18_Weights
import torch.nn as nn

# Load ImageNet pretrained ResNet18
resnet_weights = ResNet18_Weights.DEFAULT
resnet18 = models.resnet18(weights=resnet_weights)

# Freeze pretrained backbone
for param in resnet18.parameters():
    param.requires_grad = False

# Replace final layer for our 10 tomato classes
resnet18.fc = nn.Linear(
    resnet18.fc.in_features,
    len(class_names)
)

# Move model to GPU 0
resnet18 = resnet18.to(device)

print("ResNet18 loaded successfully!")
print("Pretrained weights: ImageNet")
print("Backbone: Frozen")
print("Final classification outputs:", len(class_names))
print("Training device:", next(resnet18.parameters()).device)
print(
    "Trainable parameters:",
    sum(p.numel() for p in resnet18.parameters() if p.requires_grad)
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s] 


ResNet18 loaded successfully!
Pretrained weights: ImageNet
Backbone: Frozen
Final classification outputs: 10
Training device: cuda:0
Trainable parameters: 5130


In [33]:
# Configure weighted loss
resnet_criterion = nn.CrossEntropyLoss(
    weight=weights_tensor
)

# Optimize only trainable parameters
resnet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet18.parameters()),
    lr=0.001,
    weight_decay=1e-4
)

# Training configuration
RESNET_EPOCHS = 10
BEST_RESNET_VAL_ACCURACY = 0.0

resnet_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

RESNET_CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "resnet18_checkpoints"
)

os.makedirs(RESNET_CHECKPOINT_DIR, exist_ok=True)

print("ResNet18 training configuration ready!")
print("Epochs:", RESNET_EPOCHS)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Loss: Weighted CrossEntropyLoss")
print("Checkpoint directory:", RESNET_CHECKPOINT_DIR)
print("GPU:", torch.cuda.get_device_name(0))

ResNet18 training configuration ready!
Epochs: 10
Optimizer: Adam
Learning rate: 0.001
Loss: Weighted CrossEntropyLoss
Checkpoint directory: /kaggle/working/PlantDiseaseProject/resnet18_checkpoints
GPU: Tesla T4


In [34]:
def train_resnet_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    # Keep frozen BatchNorm layers in evaluation mode
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100 * correct / total


def evaluate_resnet(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100 * correct / total


print("ResNet18 training function created successfully!")
print("ResNet18 evaluation function created successfully!")

ResNet18 training function created successfully!
ResNet18 evaluation function created successfully!


In [35]:
for epoch in range(RESNET_EPOCHS):

    print(f"\n{'='*60}")
    print(f"ResNet18 Epoch {epoch + 1}/{RESNET_EPOCHS}")
    print(f"{'='*60}")

    train_loss, train_acc = train_resnet_one_epoch(
        resnet18,
        train_loader,
        resnet_criterion,
        resnet_optimizer,
        device
    )

    val_loss, val_acc = evaluate_resnet(
        resnet18,
        val_loader,
        resnet_criterion,
        device
    )

    resnet_history["train_loss"].append(train_loss)
    resnet_history["train_accuracy"].append(train_acc)
    resnet_history["val_loss"].append(val_loss)
    resnet_history["val_accuracy"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_acc:.2f}%")

    # Save checkpoint after every epoch
    checkpoint_path = os.path.join(
        RESNET_CHECKPOINT_DIR,
        f"resnet18_epoch_{epoch + 1}.pth"
    )

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": resnet18.state_dict(),
        "optimizer_state_dict": resnet_optimizer.state_dict(),
        "history": resnet_history,
        "best_val_accuracy": max(BEST_RESNET_VAL_ACCURACY, val_acc)
    }, checkpoint_path)

    # Save best model
    if val_acc > BEST_RESNET_VAL_ACCURACY:

        BEST_RESNET_VAL_ACCURACY = val_acc

        best_path = os.path.join(
            RESNET_CHECKPOINT_DIR,
            "best_resnet18.pth"
        )

        torch.save(resnet18.state_dict(), best_path)

        print(
            f"New best ResNet18 saved! "
            f"Validation accuracy: {BEST_RESNET_VAL_ACCURACY:.2f}%"
        )

    print("Checkpoint saved:", checkpoint_path)

print("\nResNet18 training completed!")
print(
    f"Best validation accuracy: "
    f"{BEST_RESNET_VAL_ACCURACY:.2f}%"
)


ResNet18 Epoch 1/10
Train Loss: 1.1536 | Train Accuracy: 72.92%
Val Loss:   0.7019 | Val Accuracy:   82.56%
New best ResNet18 saved! Validation accuracy: 82.56%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/resnet18_checkpoints/resnet18_epoch_1.pth

ResNet18 Epoch 2/10
Train Loss: 0.6342 | Train Accuracy: 85.11%
Val Loss:   0.5533 | Val Accuracy:   86.28%
New best ResNet18 saved! Validation accuracy: 86.28%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/resnet18_checkpoints/resnet18_epoch_2.pth

ResNet18 Epoch 3/10
Train Loss: 0.5037 | Train Accuracy: 87.71%
Val Loss:   0.4498 | Val Accuracy:   88.85%
New best ResNet18 saved! Validation accuracy: 88.85%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/resnet18_checkpoints/resnet18_epoch_3.pth

ResNet18 Epoch 4/10
Train Loss: 0.4502 | Train Accuracy: 88.79%
Val Loss:   0.3884 | Val Accuracy:   90.09%
New best ResNet18 saved! Validation accuracy: 90.09%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/resnet18_c

In [36]:
# Load best ResNet18 checkpoint
best_resnet_path = os.path.join(
    RESNET_CHECKPOINT_DIR, "best_resnet18.pth"
)

resnet18.load_state_dict(
    torch.load(best_resnet_path, map_location=device, weights_only=True)
)

resnet18.eval()

resnet_predictions = []
resnet_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = resnet18(images)
        predictions = outputs.argmax(dim=1)

        resnet_predictions.extend(predictions.cpu().numpy())
        resnet_labels.extend(labels.numpy())

# Calculate metrics
resnet_test_accuracy = accuracy_score(
    resnet_labels, resnet_predictions
)

resnet_test_precision = precision_score(
    resnet_labels, resnet_predictions,
    average="macro", zero_division=0
)

resnet_test_recall = recall_score(
    resnet_labels, resnet_predictions,
    average="macro", zero_division=0
)

resnet_test_f1 = f1_score(
    resnet_labels, resnet_predictions,
    average="macro", zero_division=0
)

print("RESNET18 — TEST RESULTS")
print("=" * 50)
print(f"Test Accuracy:   {resnet_test_accuracy * 100:.2f}%")
print(f"Macro Precision: {resnet_test_precision * 100:.2f}%")
print(f"Macro Recall:    {resnet_test_recall * 100:.2f}%")
print(f"Macro F1-Score:  {resnet_test_f1 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    resnet_labels,
    resnet_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

resnet_cm = confusion_matrix(
    resnet_labels,
    resnet_predictions,
    labels=list(range(len(class_names)))
)

print("Confusion matrix shape:", resnet_cm.shape)
print("ResNet18 test evaluation completed!")

RESNET18 — TEST RESULTS
Test Accuracy:   90.95%
Macro Precision: 85.83%
Macro Recall:    88.81%
Macro F1-Score:  86.78%

CLASSIFICATION REPORT
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.89      0.97      0.93       319
                        Tomato___Early_blight       0.83      0.73      0.78       150
                         Tomato___Late_blight       0.94      0.92      0.93       210
                           Tomato___Leaf_Mold       0.71      0.88      0.79        69
                  Tomato___Septoria_leaf_spot       0.91      0.77      0.84       146
Tomato___Spider_mites Two-spotted_spider_mite       0.78      0.94      0.85       169
                         Tomato___Target_Spot       0.87      0.80      0.84       152
       Tomato___Tomato_Yellow_Leaf_Curl_Virus       0.99      0.96      0.97       623
                 Tomato___Tomato_mosaic_virus       0.68      0.96      0

In [37]:
# Save overall metrics
resnet_metrics = {
    "model": "ResNet18",
    "pretrained": "ImageNet",
    "test_accuracy": float(resnet_test_accuracy),
    "macro_precision": float(resnet_test_precision),
    "macro_recall": float(resnet_test_recall),
    "macro_f1": float(resnet_test_f1),
    "best_validation_accuracy": float(BEST_RESNET_VAL_ACCURACY),
    "epochs": RESNET_EPOCHS,
    "test_samples": len(resnet_labels)
}

with open(
    os.path.join(PROJECT_DIR, "resnet18_metrics.json"), "w"
) as f:
    json.dump(resnet_metrics, f, indent=4)

# Save classification report
resnet_report = classification_report(
    resnet_labels,
    resnet_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(resnet_report).transpose().to_csv(
    os.path.join(PROJECT_DIR, "resnet18_classification_report.csv")
)

# Save confusion matrix
pd.DataFrame(
    resnet_cm,
    index=class_names,
    columns=class_names
).to_csv(
    os.path.join(PROJECT_DIR, "resnet18_confusion_matrix.csv")
)

# Save training history
pd.DataFrame(resnet_history).to_csv(
    os.path.join(PROJECT_DIR, "resnet18_training_history.csv"),
    index=False
)

print("ResNet18 results saved successfully!")
print("Saved files:")
print("- resnet18_metrics.json")
print("- resnet18_classification_report.csv")
print("- resnet18_confusion_matrix.csv")
print("- resnet18_training_history.csv")

ResNet18 results saved successfully!
Saved files:
- resnet18_metrics.json
- resnet18_classification_report.csv
- resnet18_confusion_matrix.csv
- resnet18_training_history.csv


In [38]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import torch.nn as nn

# Load pretrained MobileNetV2
mobilenet_weights = MobileNet_V2_Weights.DEFAULT

mobilenet = mobilenet_v2(weights=mobilenet_weights)

# Freeze pretrained backbone
for param in mobilenet.parameters():
    param.requires_grad = False

# Replace final classifier layer
mobilenet.classifier[1] = nn.Linear(
    mobilenet.last_channel,
    len(class_names)
)

# Move model to GPU 0
mobilenet = mobilenet.to(device)

print("MobileNetV2 loaded successfully!")
print("Pretrained weights: ImageNet")
print("Backbone: Frozen")
print("Number of classes:", len(class_names))
print("Training device:", next(mobilenet.parameters()).device)

print(
    "Trainable parameters:",
    sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 119MB/s] 

MobileNetV2 loaded successfully!
Pretrained weights: ImageNet
Backbone: Frozen
Number of classes: 10
Training device: cuda:0
Trainable parameters: 12810


In [39]:
# Weighted loss for class imbalance
mobile_criterion = nn.CrossEntropyLoss(
    weight=weights_tensor
)

# Optimize only trainable classifier parameters
mobile_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, mobilenet.parameters()),
    lr=0.001,
    weight_decay=1e-4
)

# Training configuration
MOBILE_EPOCHS = 10
BEST_MOBILE_VAL_ACCURACY = 0.0

mobile_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

MOBILE_CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "mobilenetv2_checkpoints"
)

os.makedirs(MOBILE_CHECKPOINT_DIR, exist_ok=True)

print("MobileNetV2 training configuration ready!")
print("Epochs:", MOBILE_EPOCHS)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Loss: Weighted CrossEntropyLoss")
print("Checkpoint directory:", MOBILE_CHECKPOINT_DIR)
print("GPU:", torch.cuda.get_device_name(0))

MobileNetV2 training configuration ready!
Epochs: 10
Optimizer: Adam
Learning rate: 0.001
Loss: Weighted CrossEntropyLoss
Checkpoint directory: /kaggle/working/PlantDiseaseProject/mobilenetv2_checkpoints
GPU: Tesla T4


In [40]:
def train_mobile_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    # Keep frozen BatchNorm layers in evaluation mode
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100 * correct / total


def evaluate_mobile(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100 * correct / total


print("MobileNetV2 training function created successfully!")
print("MobileNetV2 evaluation function created successfully!")

MobileNetV2 training function created successfully!
MobileNetV2 evaluation function created successfully!


In [41]:
for epoch in range(MOBILE_EPOCHS):

    print(f"\n{'='*60}")
    print(f"MobileNetV2 Epoch {epoch + 1}/{MOBILE_EPOCHS}")
    print(f"{'='*60}")

    train_loss, train_acc = train_mobile_one_epoch(
        mobilenet,
        train_loader,
        mobile_criterion,
        mobile_optimizer,
        device
    )

    val_loss, val_acc = evaluate_mobile(
        mobilenet,
        val_loader,
        mobile_criterion,
        device
    )

    mobile_history["train_loss"].append(train_loss)
    mobile_history["train_accuracy"].append(train_acc)
    mobile_history["val_loss"].append(val_loss)
    mobile_history["val_accuracy"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_acc:.2f}%")

    # Save checkpoint after every epoch
    checkpoint_path = os.path.join(
        MOBILE_CHECKPOINT_DIR,
        f"mobilenetv2_epoch_{epoch + 1}.pth"
    )

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": mobilenet.state_dict(),
        "optimizer_state_dict": mobile_optimizer.state_dict(),
        "history": mobile_history,
        "best_val_accuracy": max(BEST_MOBILE_VAL_ACCURACY, val_acc)
    }, checkpoint_path)

    # Save best model separately
    if val_acc > BEST_MOBILE_VAL_ACCURACY:

        BEST_MOBILE_VAL_ACCURACY = val_acc

        best_path = os.path.join(
            MOBILE_CHECKPOINT_DIR,
            "best_mobilenetv2.pth"
        )

        torch.save(mobilenet.state_dict(), best_path)

        print(
            f"New best MobileNetV2 saved! "
            f"Validation accuracy: {BEST_MOBILE_VAL_ACCURACY:.2f}%"
        )

    print("Checkpoint saved:", checkpoint_path)

print("\nMobileNetV2 training completed!")
print(
    f"Best validation accuracy: "
    f"{BEST_MOBILE_VAL_ACCURACY:.2f}%"
)


MobileNetV2 Epoch 1/10
Train Loss: 1.2303 | Train Accuracy: 74.61%
Val Loss:   0.8099 | Val Accuracy:   83.75%
New best MobileNetV2 saved! Validation accuracy: 83.75%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/mobilenetv2_checkpoints/mobilenetv2_epoch_1.pth

MobileNetV2 Epoch 2/10
Train Loss: 0.6914 | Train Accuracy: 84.96%
Val Loss:   0.6055 | Val Accuracy:   86.42%
New best MobileNetV2 saved! Validation accuracy: 86.42%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/mobilenetv2_checkpoints/mobilenetv2_epoch_2.pth

MobileNetV2 Epoch 3/10
Train Loss: 0.5530 | Train Accuracy: 87.47%
Val Loss:   0.5202 | Val Accuracy:   88.47%
New best MobileNetV2 saved! Validation accuracy: 88.47%
Checkpoint saved: /kaggle/working/PlantDiseaseProject/mobilenetv2_checkpoints/mobilenetv2_epoch_3.pth

MobileNetV2 Epoch 4/10
Train Loss: 0.4787 | Train Accuracy: 88.67%
Val Loss:   0.4482 | Val Accuracy:   90.81%
New best MobileNetV2 saved! Validation accuracy: 90.81%
Checkpoint saved: /kag

In [42]:
# Load best MobileNetV2 checkpoint
best_mobile_path = os.path.join(
    MOBILE_CHECKPOINT_DIR, "best_mobilenetv2.pth"
)

mobilenet.load_state_dict(
    torch.load(best_mobile_path, map_location=device, weights_only=True)
)

mobilenet.eval()

mobile_predictions = []
mobile_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = mobilenet(images)
        predictions = outputs.argmax(dim=1)

        mobile_predictions.extend(predictions.cpu().numpy())
        mobile_labels.extend(labels.numpy())

# Calculate test metrics
mobile_test_accuracy = accuracy_score(
    mobile_labels, mobile_predictions
)

mobile_test_precision = precision_score(
    mobile_labels, mobile_predictions,
    average="macro", zero_division=0
)

mobile_test_recall = recall_score(
    mobile_labels, mobile_predictions,
    average="macro", zero_division=0
)

mobile_test_f1 = f1_score(
    mobile_labels, mobile_predictions,
    average="macro", zero_division=0
)

print("MOBILENETV2 — TEST RESULTS")
print("=" * 50)
print(f"Test Accuracy:   {mobile_test_accuracy * 100:.2f}%")
print(f"Macro Precision: {mobile_test_precision * 100:.2f}%")
print(f"Macro Recall:    {mobile_test_recall * 100:.2f}%")
print(f"Macro F1-Score:  {mobile_test_f1 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    mobile_labels,
    mobile_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

mobile_cm = confusion_matrix(
    mobile_labels,
    mobile_predictions,
    labels=list(range(len(class_names)))
)

print("Confusion matrix shape:", mobile_cm.shape)
print("MobileNetV2 test evaluation completed!")

MOBILENETV2 — TEST RESULTS
Test Accuracy:   92.14%
Macro Precision: 88.39%
Macro Recall:    90.07%
Macro F1-Score:  88.97%

CLASSIFICATION REPORT
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.96      0.93      0.95       319
                        Tomato___Early_blight       0.82      0.77      0.80       150
                         Tomato___Late_blight       0.91      0.96      0.94       210
                           Tomato___Leaf_Mold       0.93      0.81      0.87        69
                  Tomato___Septoria_leaf_spot       0.83      0.89      0.86       146
Tomato___Spider_mites Two-spotted_spider_mite       0.84      0.94      0.89       169
                         Tomato___Target_Spot       0.82      0.77      0.79       152
       Tomato___Tomato_Yellow_Leaf_Curl_Virus       0.99      0.97      0.98       623
                 Tomato___Tomato_mosaic_virus       0.77      1.00    

In [43]:
# Save MobileNetV2 metrics
mobile_metrics = {
    "model": "MobileNetV2",
    "pretrained": "ImageNet",
    "test_accuracy": float(mobile_test_accuracy),
    "macro_precision": float(mobile_test_precision),
    "macro_recall": float(mobile_test_recall),
    "macro_f1": float(mobile_test_f1),
    "best_validation_accuracy": float(BEST_MOBILE_VAL_ACCURACY),
    "epochs": MOBILE_EPOCHS,
    "test_samples": len(mobile_labels)
}

with open(
    os.path.join(PROJECT_DIR, "mobilenetv2_metrics.json"), "w"
) as f:
    json.dump(mobile_metrics, f, indent=4)

# Save classification report
mobile_report = classification_report(
    mobile_labels,
    mobile_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(mobile_report).transpose().to_csv(
    os.path.join(PROJECT_DIR, "mobilenetv2_classification_report.csv")
)

# Save confusion matrix
pd.DataFrame(
    mobile_cm,
    index=class_names,
    columns=class_names
).to_csv(
    os.path.join(PROJECT_DIR, "mobilenetv2_confusion_matrix.csv")
)

# Save training history
pd.DataFrame(mobile_history).to_csv(
    os.path.join(PROJECT_DIR, "mobilenetv2_training_history.csv"),
    index=False
)

# Create final three-model comparison
comparison = pd.DataFrame([
    {
        "Model": "Custom CNN",
        "Test Accuracy (%)": test_accuracy * 100,
        "Macro Precision (%)": test_precision * 100,
        "Macro Recall (%)": test_recall * 100,
        "Macro F1 (%)": test_f1 * 100,
        "Best Validation Accuracy (%)": BEST_VAL_ACCURACY
    },
    {
        "Model": "ResNet18",
        "Test Accuracy (%)": resnet_test_accuracy * 100,
        "Macro Precision (%)": resnet_test_precision * 100,
        "Macro Recall (%)": resnet_test_recall * 100,
        "Macro F1 (%)": resnet_test_f1 * 100,
        "Best Validation Accuracy (%)": BEST_RESNET_VAL_ACCURACY
    },
    {
        "Model": "MobileNetV2",
        "Test Accuracy (%)": mobile_test_accuracy * 100,
        "Macro Precision (%)": mobile_test_precision * 100,
        "Macro Recall (%)": mobile_test_recall * 100,
        "Macro F1 (%)": mobile_test_f1 * 100,
        "Best Validation Accuracy (%)": BEST_MOBILE_VAL_ACCURACY
    }
])

comparison.to_csv(
    os.path.join(PROJECT_DIR, "classification_model_comparison.csv"),
    index=False
)

print("MobileNetV2 results saved successfully!")
print("\nFINAL MODEL COMPARISON")
display(comparison.round(2))

print("\nBest model by test accuracy:",
      comparison.loc[comparison["Test Accuracy (%)"].idxmax(), "Model"])

print("\nAll classification results saved in:", PROJECT_DIR)

MobileNetV2 results saved successfully!

FINAL MODEL COMPARISON


,Model,Test Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Best Validation Accuracy (%)
0,Custom CNN,90.48,85.39,90.59,86.82,90.52
1,ResNet18,90.95,85.83,88.81,86.78,91.52
2,MobileNetV2,92.14,88.39,90.07,88.97,92.42



Best model by test accuracy: MobileNetV2

All classification results saved in: /kaggle/working/PlantDiseaseProject


In [44]:

import os

print("PROJECT FILES AND MODEL CHECKPOINTS\n")

for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        if file.endswith((".pth", ".pt", ".json", ".csv")):
            print(f"{indent}    {file}")


PROJECT FILES AND MODEL CHECKPOINTS

PlantDiseaseProject/
    mobilenetv2_training_history.csv
    resnet18_metrics.json
    class_weights.pt
    custom_cnn_classification_report.csv
    class_mapping.json
    mobilenetv2_confusion_matrix.csv
    resnet18_classification_report.csv
    mobilenetv2_metrics.json
    custom_cnn_training_history.csv
    classification_model_comparison.csv
    validation_manifest.csv
    custom_cnn_metrics.json
    resnet18_confusion_matrix.csv
    train_manifest.csv
    resnet18_training_history.csv
    mobilenetv2_classification_report.csv
    custom_cnn_confusion_matrix.csv
    test_manifest.csv
    checkpoints/
        custom_cnn_epoch_6.pth
        custom_cnn_epoch_7.pth
        custom_cnn_epoch_10.pth
        custom_cnn_epoch_2.pth
        custom_cnn_epoch_4.pth
        custom_cnn_epoch_8.pth
        best_custom_cnn.pth
        custom_cnn_epoch_5.pth
        custom_cnn_epoch_1.pth
        custom_cnn_epoch_9.pth
        custom_cnn_epoch_3.pth
    resnet

In [45]:

import os
import torch
import json

checkpoint_path = os.path.join(
    PROJECT_DIR,
    "mobilenetv2_checkpoints",
    "best_mobilenetv2.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("CHECKPOINT TYPE:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCHECKPOINT KEYS:")
    for key in checkpoint.keys():
        print("-", key)

    print("\nMODEL WEIGHT DETAILS:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        elif torch.is_tensor(value):
            print(f"{key}: tensor, shape={tuple(value.shape)}")

mapping_path = os.path.join(PROJECT_DIR, "class_mapping.json")

with open(mapping_path, "r") as f:
    saved_class_mapping = json.load(f)

print("\nCLASS MAPPING:")
print(saved_class_mapping)


CHECKPOINT TYPE: <class 'collections.OrderedDict'>

CHECKPOINT KEYS:
- features.0.0.weight
- features.0.1.weight
- features.0.1.bias
- features.0.1.running_mean
- features.0.1.running_var
- features.0.1.num_batches_tracked
- features.1.conv.0.0.weight
- features.1.conv.0.1.weight
- features.1.conv.0.1.bias
- features.1.conv.0.1.running_mean
- features.1.conv.0.1.running_var
- features.1.conv.0.1.num_batches_tracked
- features.1.conv.1.weight
- features.1.conv.2.weight
- features.1.conv.2.bias
- features.1.conv.2.running_mean
- features.1.conv.2.running_var
- features.1.conv.2.num_batches_tracked
- features.2.conv.0.0.weight
- features.2.conv.0.1.weight
- features.2.conv.0.1.bias
- features.2.conv.0.1.running_mean
- features.2.conv.0.1.running_var
- features.2.conv.0.1.num_batches_tracked
- features.2.conv.1.0.weight
- features.2.conv.1.1.weight
- features.2.conv.1.1.bias
- features.2.conv.1.1.running_mean
- features.2.conv.1.1.running_var
- features.2.conv.1.1.num_batches_tracked
- fea

In [46]:

import os
import shutil

PROJECT_DIR = "/kaggle/working/PlantDiseaseProject"
ZIP_BASE = "/kaggle/working/PlantDiseaseProject_Backup"

backup_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir="/kaggle/working",
    base_dir="PlantDiseaseProject"
)

print("BACKUP CREATED SUCCESSFULLY!")
print("File:", backup_path)
print("Size:", round(os.path.getsize(backup_path) / (1024**2), 2), "MB")


BACKUP CREATED SUCCESSFULLY!
File: /kaggle/working/PlantDiseaseProject_Backup.zip
Size: 569.58 MB


In [48]:
import os
from IPython.display import display, FileLink

zip_path = "/kaggle/working/PlantDiseaseProject_Backup.zip"

print("ZIP exists:", os.path.exists(zip_path))
if os.path.exists(zip_path):
    print("Size:", round(os.path.getsize(zip_path) / (1024**2), 2), "MB")
    display(FileLink(zip_path))
else:
    print("ZIP not found!")

ZIP exists: True
Size: 569.58 MB


/kaggle/working/PlantDiseaseProject_Backup.zip